<a href="https://colab.research.google.com/github/dipeshMahakali/Sarthika-AI/blob/main/AGI_Cognitive_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Autonomous Cognitive Architecture (Micro-AGI Engine)
### Google Colab GPU-Accelerated Implementation

---

### What is Demonstrated in this Architecture?
In modern artificial intelligence research, achieving **AGI-level functionality** requires far more than next-token language generation. It requires an **embodied cognitive loop** that possesses:

1. **Tripartite Memory Architecture**:
   - **Working Memory**: Dynamic scratchpad, active sub-task tree, and attention budgeting.
   - **Episodic Memory**: Vector-indexed recall (FAISS + SentenceTransformers) of past actions, successes, and failures for continuous lifelong learning.
   - **Semantic Memory**: Relational SQLite knowledge base for persistent facts and domain rules.
   - **Procedural Memory (Skills)**: Self-directed code synthesis where the agent creates new tools, validates them in a sandbox, and registers them dynamically (Voyager/Eureka paradigm).

2. **Dual-Process Cognition (System 1 & System 2)**:
   - **System 1**: Fast heuristic generation and intuitive responses.
   - **System 2**: Deliberate Tree-of-Thought decomposition, constraint checking, and metacognitive self-critique.

3. **Autonomous Execution Sandbox**:
   - Live Python REPL and shell execution with automatic error trapping, traceback analysis, and self-debugging.

4. **Hardware Optimized for Colab**:
   - Runs `Qwen/Qwen2.5-7B-Instruct` (or `Llama-3.1-8B-Instruct`) in **4-bit NormalFloat (NF4)** using `bitsandbytes` on a standard Colab T4 or A100 GPU.


## 📦 Step 1: Install Dependencies & Check Hardware Acceleration
This cell installs the necessary libraries:
- `transformers`, `accelerate`, `bitsandbytes` for 4-bit GPU model inference
- `sentence-transformers` & `faiss-cpu` for vector episodic memory
- `rich` for formatted cognitive tracing


In [1]:
#@title Install Dependencies and Verify GPU
import os
import sys

print("Installing cognitive architecture dependencies...")
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu rich pydantic

import torch
print("" + "="*50)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU DETECTED: {device_name}")
    print(f"✅ VRAM AVAILABLE: {vram_gb:.2f} GB")
    device = "cuda"
else:
    print("⚠️ NO GPU DETECTED! Running on CPU mode (Execution will be slower).")
    print("👉 To enable GPU: Click Runtime -> Change runtime type -> Select T4 GPU.")
    device = "cpu"
print("="*50)


Installing cognitive architecture dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00
✅ GPU DETECTED: Tesla T4
✅ VRAM AVAILABLE: 14.56 GB


## ⚡ Step 2: The Cognitive Reasoning Engine
Loads the foundation reasoning model (`Qwen/Qwen2.5-7B-Instruct`) in 4-bit quantization, enabling high-performance local inference on Colab's 16GB T4 GPU.
A pluggable API fallback (Groq / Gemini / OpenAI) is also included for zero-VRAM execution if desired.


In [2]:
#@title Initialize Reasoning Engine
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

class CognitiveEngine:
    def __init__(self, model_id="Qwen/Qwen2.5-7B-Instruct", use_4bit=True):
        self.model_id = model_id
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Loading Cognitive Engine: {model_id} on {self.device}...")

        if self.device == "cuda" and use_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_use_double_quant=True
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto",
                trust_remote_code=True
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_id,
                torch_dtype=torch.float32 if self.device == "cpu" else torch.float16,
                device_map="auto",
                trust_remote_code=True
            )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        print("✅ Cognitive Engine successfully loaded into memory!")

    def generate(self, prompt: str, system_prompt: str = "You are an advanced AGI cognitive reasoning core.", max_new_tokens: int = 1024, temperature: float = 0.2):
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        with torch.no_grad():
            generated_ids = self.model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                do_sample=True if temperature > 0 else False,
                pad_token_id=self.tokenizer.eos_token_id
            )

        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        return self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Instantiate model
engine = CognitiveEngine()


Loading Cognitive Engine: Qwen/Qwen2.5-7B-Instruct on cuda...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Cognitive Engine successfully loaded into memory!


## 🧠 Step 3: Tripartite Memory Subsystem
Humans solve novel problems by combining:
1. **Working Memory**: In-context task state and scratchpad.
2. **Semantic Memory**: Structured facts, relationships, and domain ontologies (stored in SQLite).
3. **Episodic Memory**: Experiential memory of past successes, failures, and reflections (stored as vector embeddings with FAISS).


In [7]:
#@title Build Tripartite Memory Architecture
import sqlite3
import json
import numpy as np
from typing import List, Dict, Any, Optional
from sentence_transformers import SentenceTransformer
import faiss

# --- 1. Working Memory ---
class WorkingMemory:
    def __init__(self):
        self.objective: str = ""
        self.scratchpad: List[str] = []
        self.subtasks: List[Dict[str, Any]] = []
        self.current_step: int = 0

    def reset(self, objective: str):
        self.objective = objective
        self.scratchpad = []
        self.subtasks = []
        self.current_step = 0

    def add_thought(self, thought: str):
        self.scratchpad.append(f"[Step {self.current_step}] {thought}")

    def get_summary(self) -> str:
        recent_thoughts = "\n".join(self.scratchpad[-5:])
        return f"""Objective: {self.objective}
Recent Thoughts:
{recent_thoughts}"""

# --- 2. Semantic Memory (SQLite Structured Knowledge) ---
class SemanticMemory:
    def __init__(self, db_path=":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_db()

    def _init_db(self):
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS facts (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    subject TEXT,
                    predicate TEXT,
                    object TEXT,
                    confidence REAL DEFAULT 1.0,
                    source TEXT
                )
            """)

    def store_fact(self, subject: str, predicate: str, obj: str, source: str = "agent_discovery"):
        with self.conn:
            self.conn.execute(
                "INSERT INTO facts (subject, predicate, object, source) VALUES (?, ?, ?, ?)",
                (subject.lower(), predicate.lower(), obj, source)
            )

    def query_facts(self, entity: str) -> List[str]:
        cur = self.conn.cursor()
        cur.execute("SELECT subject, predicate, object FROM facts WHERE subject LIKE ? OR object LIKE ?", (f"%{entity.lower()}%", f"%{entity.lower()}% "))
        results = cur.fetchall()
        return [f"{sub} {pred} {obj}" for sub, pred, obj in results]

# --- 3. Episodic Memory (Vector Experiential Recall) ---
class EpisodicMemory:
    def __init__(self, embedder_model="all-MiniLM-L6-v2"):
        print("Loading embedding model for Episodic Memory...")
        self.embedder = SentenceTransformer(embedder_model)
        self.dim = self.embedder.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatIP(self.dim)
        self.episodes: List[Dict[str, Any]] = []

    def record_episode(self, task: str, action: str, result: str, success: bool, reflection: str):
        episode = {
            "task": task,
            "action": action,
            "result": result[:500],
            "success": success,
            "reflection": reflection
        }
        text_repr = f"Task: {task} | Action: {action} | Outcome: {'Success' if success else 'Failure'} | Lesson: {reflection}"
        emb = self.embedder.encode([text_repr], normalize_embeddings=True).astype("float32")
        self.index.add(emb)
        self.episodes.append(episode)

    def recall_similar(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        if self.index.ntotal == 0:
            return []
        q_emb = self.embedder.encode([query], normalize_embeddings=True).astype("float32")
        k = min(top_k, self.index.ntotal)
        distances, indices = self.index.search(q_emb, k)
        return [self.episodes[idx] for idx in indices[0] if idx != -1]

# Initialize Memories
working_mem = WorkingMemory()
semantic_mem = SemanticMemory()
episodic_mem = EpisodicMemory()
print("✅ Tripartite Memory Subsystem initialized successfully!")


Loading embedding model for Episodic Memory...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Tripartite Memory Subsystem initialized successfully!


/tmp/ipykernel_1060/2984062658.py:69: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.embedder.get_sentence_embedding_dimension()


## 🛠️ Step 4: Procedural Skill Synthesis & Execution Sandbox
A fundamental hallmark of general intelligence is **tool synthesis (self-programming)**.
Instead of having a fixed set of hardcoded tools, the agent can:
1. Write raw Python code to implement any new capability.
2. Execute and test the code inside a protected execution environment.
3. Automatically capture standard output, return values, or syntax/runtime tracebacks.
4. Once verified, permanently register the new tool in its **Procedural Memory**.


In [8]:
#@title Sandbox Executor & Procedural Skill Registry
import io
import sys
import traceback

class SandboxExecutor:
    @staticmethod
    def execute_python(code: str, timeout_seconds: int = 10) -> Dict[str, Any]:
        """Executes code in a controlled namespace capturing stdout, stderr, and variables."""
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        redirected_output = io.StringIO()
        redirected_error = io.StringIO()

        sys.stdout = redirected_output
        sys.stderr = redirected_error

        exec_globals = {"__builtins__": __builtins__}
        exec_locals = {}

        success = False
        output_str = ""
        error_str = ""

        try:
            exec(code, exec_globals, exec_locals)
            success = True
            output_str = redirected_output.getvalue()
        except Exception as e:
            error_str = traceback.format_exc()
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

        return {
            "success": success,
            "stdout": output_str,
            "error": error_str,
            "return_state": {k: str(v) for k, v in exec_locals.items() if not k.startswith("_")}
        }

class SkillManager:
    def __init__(self, sandbox: SandboxExecutor):
        self.sandbox = sandbox
        self.skills: Dict[str, Dict[str, Any]] = {}
        self._register_default_skills()

    def _register_default_skills(self):
        self.register_skill(
            name="calculator",
            docstring="Performs exact mathematical evaluation.",
            code="""
def calculator(expr: str):
    import math
    return eval(expr, {"__builtins__": None, "math": math})
"""
        )

    def register_skill(self, name: str, docstring: str, code: str):
        # Validate syntax by compiling
        try:
            compile(code, f"<skill_{name}>", "exec")
            self.skills[name] = {
                "name": name,
                "docstring": docstring,
                "code": code
            }
            return True, f"Skill '{name}' registered successfully."
        except Exception as e:
            return False, f"Failed to register skill '{name}': {e}"

    def get_skill_docs(self) -> str:
        docs = []
        for name, data in self.skills.items():
            docs.append(f"- **{name}**: {data['docstring']}")
        return "\n".join(docs)

    def execute_skill(self, name: str, *args, **kwargs) -> Any:
        if name not in self.skills:
            raise ValueError(f"Skill '{name}' does not exist.")
        skill_code = self.skills[name]["code"]
        namespace = {}
        exec(skill_code, namespace)
        fn = namespace[name]
        return fn(*args, **kwargs)

sandbox = SandboxExecutor()
skill_manager = SkillManager(sandbox)
print("✅ Sandbox & Procedural Skill Registry ready!")
print("Initial skills:\n" + skill_manager.get_skill_docs())


✅ Sandbox & Procedural Skill Registry ready!
Initial skills:
- **calculator**: Performs exact mathematical evaluation.


## 🔬 Step 5: System 2 Metacognition & Self-Critique
The Metacognitive module conducts:
- **Hierarchical Goal Planning**: Breaks an ambitious objective into discrete, verifiable milestones.
- **Reflection / Self-Critique**: Inspects observations, catches logical flaws or execution errors, and formulates recovery strategies.


In [9]:
#@title Metacognitive Planner and Critic
import re

class MetacognitivePlanner:
    def __init__(self, engine: CognitiveEngine):
        self.engine = engine

    def decompose_objective(self, objective: str, memory_context: str) -> List[str]:
        prompt = f"""Given the following high-level objective and relevant memory context, break it down into an optimal sequence of 2 to 4 concrete, actionable sub-tasks.
Objective: {objective}

Memory Context:
{memory_context}

Output only a numbered list of sub-tasks (1. ..., 2. ..., etc.):"""
        response = self.engine.generate(
            prompt=prompt,
            system_prompt="You are a System 2 Metacognitive Task Planner. You produce lean, precise, execution-oriented sub-plans."
        )

        # Parse numbered tasks
        tasks = []
        for line in response.strip().split("\n"):
            match = re.match(r"^\d+\.\s*(.*)", line.strip())
            if match:
                tasks.append(match.group(1).strip())
        return tasks if tasks else [objective]

class MetacognitiveCritic:
    def __init__(self, engine: CognitiveEngine):
        self.engine = engine

    def evaluate_step(self, subtask: str, action: str, observation: str) -> Dict[str, Any]:
        prompt = f"""Critique the outcome of this action against the desired sub-task.
Subtask: {subtask}
Action Executed: {action}
Execution Observation: {observation}

Respond in the following format:
SUCCESS: <YES or NO>
CRITIQUE: <one sentence analysis of what happened>
NEXT_RECOMMENDATION: <what the agent should do next>"""

        response = self.engine.generate(
            prompt=prompt,
            system_prompt="You are an objective Metacognitive Critic verifying factual execution accuracy."
        )

        success = "SUCCESS: YES" in response.upper()
        critique = "No critique available."
        next_rec = "Continue to next step."

        for line in response.split("\n"):
            if line.startswith("CRITIQUE:"):
                critique = line.replace("CRITIQUE:", "").strip()
            elif line.startswith("NEXT_RECOMMENDATION:"):
                next_rec = line.replace("NEXT_RECOMMENDATION:", "").strip()

        return {
            "success": success,
            "critique": critique,
            "next_recommendation": next_rec,
            "raw_response": response
        }

planner = MetacognitivePlanner(engine)
critic = MetacognitiveCritic(engine)
print("✅ Metacognitive Planner and Critic initialized.")


✅ Metacognitive Planner and Critic initialized.


## 🔄 Step 6: The Autonomous Cognitive Agent Loop
Here, all systems fuse into a continuous, self-improving execution cycle:

$$\text{Perception} \longrightarrow \text{Memory Recall} \longrightarrow \text{Plan} \longrightarrow \text{Action/Synthesis} \longrightarrow \text{Observation} \longrightarrow \text{Critique} \longrightarrow \text{Consolidation}$$


In [11]:
#@title The Autonomous Cognitive Agent
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown
import re

console = Console()

class AutonomousCognitiveAgent:
    def __init__(self, engine, working_mem, semantic_mem, episodic_mem, skill_mgr, sandbox, planner, critic):
        self.engine = engine
        self.working_mem = working_mem
        self.semantic_mem = semantic_mem
        self.episodic_mem = episodic_mem
        self.skill_mgr = skill_mgr
        self.sandbox = sandbox
        self.planner = planner
        self.critic = critic

    def run(self, objective: str, max_steps: int = 5):
        console.print(Panel.fit(f"[bold cyan]🎯 NEW AUTONOMOUS GOAL:[/bold cyan] {objective}", border_style="cyan"))

        self.working_mem.reset(objective)

        # 1. Recall from Episodic Memory (Transfer Learning from past experiences)
        past_episodes = self.episodic_mem.recall_similar(objective, top_k=2)
        memory_context = ""
        if past_episodes:
            memory_context += "Past Relevant Experiences:\n"
            for ep in past_episodes:
                memory_context += f"- Task: {ep['task']} | Result: {'Success' if ep['success'] else 'Fail'} | Lesson: {ep['reflection']}\n"

        # 2. Decompose Goal into Hierarchical Sub-tasks
        console.print("[bold yellow]🧠 System 2 Planning: Decomposing goal...[/bold yellow]")
        subtasks = self.planner.decompose_objective(objective, memory_context)
        self.working_mem.subtasks = [{"title": t, "done": False} for t in subtasks]

        for i, t in enumerate(subtasks, 1):
            console.print(f"  [green]{i}.[/green] {t}")

        # 3. Execute Sub-tasks with ReAct + Self-Critique Loop
        for step_idx, subtask_obj in enumerate(self.working_mem.subtasks, 1):
            subtask = subtask_obj["title"]
            console.print(f"\n[bold magenta]━━━━━━━━━━ STEP {step_idx}: {subtask} ━━━━━━━━━━[/bold magenta]")
            self.working_mem.current_step = step_idx

            # Formulate Action (System 1 + Tool Synthesis)
            available_skills = self.skill_mgr.get_skill_docs()
            action_prompt = f"""Current Sub-task: {subtask}
Overall Objective: {objective}

Available Skills:
{available_skills}

You have two choices for this step:
Option A: WRITE A PYTHON SCRIPT to solve, compute, or simulate this step directly.
Wrap Python code in ```python ... ```
Option B: PROVIDE THE DIRECT LOGICAL DEDUCTION.

Decide and respond:"""

            thought_and_action = self.engine.generate(
                prompt=action_prompt,
                system_prompt="You are an autonomous cognitive agent reasoning to accomplish your sub-goal."
            )

            console.print(f"[bold blue]🤔 Thought & Proposed Action:[/bold blue]\n{thought_and_action[:400]}...")

            # Check if Python code was generated
            code_match = re.search(r"```python(.*?)```", thought_and_action, re.DOTALL)
            observation = ""
            action_taken = ""

            if code_match:
                code_to_run = code_match.group(1).strip()
                action_taken = f"Executed Python Code:\n{code_to_run}"
                console.print("[yellow]⚙️ Executing sandbox code...[/yellow]")

                exec_result = self.sandbox.execute_python(code_to_run)
                if exec_result["success"]:
                    observation = f"STDOUT: {exec_result['stdout']}\nSTATE: {exec_result['return_state']}"
                    console.print(f"[green]✓ Execution Success:[/green]\n{observation[:300]}")

                    # If this code defined a reusable function, auto-register as a skill
                    fn_match = re.search(r"def\s+([a-zA-Z_][a-zA-Z0-9_]*)\(", code_to_run)
                    if fn_match:
                        fn_name = fn_match.group(1)
                        self.skill_mgr.register_skill(
                            name=fn_name,
                            docstring=f"Autonomous skill synthesized for subtask: {subtask}",
                            code=code_to_run
                        )
                        console.print(f"[bold green]✨ NEW PROCEDURAL SKILL SYNTHESIZED & REGISTERED: '{fn_name}'[/bold green]")
                else:
                    observation = f"ERROR TRACEBACK:\n{exec_result['error']}"
                    console.print(f"[bold red]✗ Execution Error:[/bold red]\n{observation}")
            else:
                action_taken = "Direct Cognitive Deduction"
                observation = thought_and_action

            # 4. System 2 Self-Critique
            console.print("[yellow]🔍 Metacognitive Self-Critique in progress...[/yellow]")
            critique_res = self.critic.evaluate_step(subtask, action_taken, observation)
            console.print(f"  [bold]Success:[/bold] {critique_res['success']}")
            console.print(f"  [bold]Critique:[/bold] {critique_res['critique']}")
            console.print(f"  [bold]Next Step:[/bold] {critique_res['next_recommendation']}")

            # 5. Consolidate into Episodic Memory (Lifelong Learning)
            self.episodic_mem.record_episode(
                task=subtask,
                action=action_taken[:300],
                result=observation[:300],
                success=critique_res["success"],
                reflection=critique_res["critique"]
            )

            subtask_obj["done"] = True

        console.print(Panel.fit("[bold green]🏁 GOAL EXECUTION FINISHED! Episodic and Procedural memories consolidated.[/bold green]", border_style="green"))

# Instantiate Agent using the correct variable name: skill_manager
agent = AutonomousCognitiveAgent(
    engine=engine,
    working_mem=working_mem,
    semantic_mem=semantic_mem,
    episodic_mem=episodic_mem,
    skill_mgr=skill_manager,
    sandbox=sandbox,
    planner=planner,
    critic=critic
)
print("✅ Autonomous Cognitive Agent is online and ready.")


✅ Autonomous Cognitive Agent is online and ready.


## 🧪 Experiment 1: Autonomous Algorithmic Discovery & Skill Synthesis
Watch the agent receive an algorithmic problem:
1. Deconstruct the problem.
2. Synthesize an algorithm in Python.
3. Test and execute in the sandbox.
4. Self-critique its logic.
5. Save the working function permanently into its **Procedural Skill Library**.


In [12]:
#@title Run Experiment 1: Algorithmic Discovery
goal = """Design an efficient algorithm to compute the nth Fibonacci number in O(log n) using matrix exponentiation, verify it for n=50, and output the result."""
agent.run(goal)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🎯 NEW AUTONOMOUS GOAL: Design an efficient algorithm to compute the nth Fibonacci number in O(log n) using     │
│ matrix exponentiation, verify it for n=50, and output the result.                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🧠 System 2 Planning: Decomposing goal...

1. Define the matrix representation for the Fibonacci sequence and the corresponding matrix exponentiation 
method.

2. Implement a function to perform matrix exponentiation using the divide-and-conquer approach.

3. Write a function to compute the nth Fibonacci number using the matrix exponentiation method.

...

🔍 Metacognitive Self-Critique in progress...

Success: False

Critique: The action taken was correct in theory but the execution resulted in an incomplete and incorrect matrix
exponentiation process.

Next Step: Complete the matrix exponentiation for \( A^{49} \) by following the binary representation method 
correctly and then compute \( F_{50} \) from the resulting matrix.

╭────────────────────────────────────────────────────────────────────────────╮
│ 🏁 GOAL EXECUTION FINISHED! Episodic and Procedural memories consolidated. │
╰────────────────────────────────────────────────────────────────────────────╯

## 🧪 Experiment 2: Lifelong Transfer Learning
In this experiment, notice how the agent:
1. Recalls past lessons from **Episodic Memory**.
2. Reuses the new procedural skills it synthesized in Experiment 1 to solve a new challenge without starting from scratch.


In [13]:
#@title Run Experiment 2: Transfer Learning and Skill Reuse
goal_2 = """Using your previously learned skills and memory, verify whether the 50th Fibonacci number is divisible by 5, and state your reasoning."""
agent.run(goal_2)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🎯 NEW AUTONOMOUS GOAL: Using your previously learned skills and memory, verify whether the 50th Fibonacci      │
│ number is divisible by 5, and state your reasoning.                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🧠 System 2 Planning: Decomposing goal...

1. Import necessary libraries (ensure numpy is available).

2. Define a function `matrix_pow` to perform matrix exponentiation.

3. Implement a function `fibonacci` that uses `matrix_pow` to compute the nth Fibonacci number.

...

🔍 Metacognitive Self-Critique in progress...

Success: True

Critique: The reasoning provided is accurate and correctly identifies the 50th Fibonacci number as being 
divisible by 5 using the Pisano period for modulo 5.

Next Step: No further action is needed; the task has been successfully completed.

╭────────────────────────────────────────────────────────────────────────────╮
│ 🏁 GOAL EXECUTION FINISHED! Episodic and Procedural memories consolidated. │
╰────────────────────────────────────────────────────────────────────────────╯

## 🚀 Step 7: Interactive Cognitive Terminal
Type **any arbitrary autonomous goal** below and observe the cognitive architecture plan, write code, self-correct, and memorize the experience.


In [14]:
#@title 🎮 Interactive Autonomous Goal Terminal
#@markdown Enter any high-level objective for the cognitive agent:
user_goal = "Analyze the Collatz Conjecture for the number 27, calculate its peak value and stopping time, and synthesize a reusable tool for it." #@param {type:"string"}

if user_goal.strip():
    agent.run(user_goal)
else:
    print("Please enter a valid goal.")


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🎯 NEW AUTONOMOUS GOAL: Analyze the Collatz Conjecture for the number 27, calculate its peak value and stopping │
│ time, and synthesize a reusable tool for it.                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🧠 System 2 Planning: Decomposing goal...

1. Define a function `collatz_sequence(n)` that takes an integer `n` and returns the Collatz sequence starting 
from `n`.

2. Modify `collatz_sequence(n)` to track and return the peak value in the sequence.

3. Add a counter to `collatz_sequence(n)` to track the stopping time (number of steps) to reach 1.

...

🔍 Metacognitive Self-Critique in progress...

Success: True

Critique: The code executed correctly and documented the `CollatzAnalyzer` class as intended.

Next Step: Test the `CollatzAnalyzer` class with various inputs to ensure its functionality and consider 
documenting any additional methods or attributes that might be useful.

╭────────────────────────────────────────────────────────────────────────────╮
│ 🏁 GOAL EXECUTION FINISHED! Episodic and Procedural memories consolidated. │
╰────────────────────────────────────────────────────────────────────────────╯